In [1]:
import pandas as pd
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import src.config as config
import src.profile_model_registry as profile_model_registry
import src.profile_tracker as profile_tracker
import src.profile_trainer as profile_trainer

importlib.reload(config)
importlib.reload(profile_model_registry)
importlib.reload(profile_tracker)
importlib.reload(profile_trainer)

from src.profile_trainer import run_profile_training_experiment

print(profile_model_registry.get_profile_dense_models().keys())
print(profile_model_registry.get_profile_nan_friendly_models().keys())

dict_keys(['ridge', 'random_forest', 'xgboost', 'catboost'])
dict_keys(['hist_gbr', 'xgboost'])


    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",

In [2]:
path = r"C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\15_min\df_BU_TotActPwr_Tech_Room.parquet"
df_imputed = pd.read_parquet(path)

In [3]:
df_imputed.head()

,BU_TotActPwr_Tech_Room,BA_Soc,PV_WS_AirTemp,PV_WS_Radiation,dayofweek,is_weekend,hour,month,dayofmonth,minute_of_day,...,BU_TotActPwr_Tech_Room_roll_std_96,BU_TotActPwr_Tech_Room_trend_1_4,BU_TotActPwr_Tech_Room_trend_1_12,BU_TotActPwr_Tech_Room_trend_1_96,BU_TotActPwr_Tech_Room_trend_96_192,BU_TotActPwr_Tech_Room_yesterday_mean,BU_TotActPwr_Tech_Room_yesterday_max,BU_TotActPwr_Tech_Room_yesterday_midday_max,BU_TotActPwr_Tech_Room_yesterday_peak_time_step,BU_TotActPwr_Tech_Room_lastweek_same_day_peak
Time,,,,,,,,,,,,,,,,,,,,,
2025-10-18 00:00:00,3.64633,81.00000,143.00000,-1.73333,5,1,0,10,18,0,...,1.495863,0.03567,0.02033,0.02100,-0.09167,2.12266,3.689,0.54267,78.0,NaN
2025-10-18 00:15:00,3.62200,80.66667,143.00000,-1.50000,5,1,0,10,18,15,...,1.495802,-0.01700,-0.03734,-0.04000,0.02100,2.12266,3.689,0.54267,78.0,NaN
2025-10-18 00:30:00,3.62467,80.20000,142.66667,-1.73333,5,1,0,10,18,30,...,1.495108,-0.01567,-0.05100,-0.04433,-0.00200,2.12266,3.689,0.54267,78.0,NaN
2025-10-18 00:45:00,3.67833,79.83333,141.66667,-2.16667,5,1,0,10,18,45,...,1.494661,-0.04833,-0.01000,0.00034,-0.06367,2.12266,3.689,0.54267,78.0,NaN
2025-10-18 01:00:00,3.63567,79.40000,140.33333,-3.13333,5,1,1,10,18,60,...,1.495243,0.03200,0.00033,-0.00534,0.03534,2.12266,3.689,0.54267,78.0,NaN


In [5]:
df_imputed.columns.tolist()

['BU_TotActPwr_Tech_Room',
 'BA_Soc',
 'PV_WS_AirTemp',
 'PV_WS_Radiation',
 'dayofweek',
 'is_weekend',
 'hour',
 'month',
 'dayofmonth',
 'minute_of_day',
 'sin_tod',
 'cos_tod',
 'sin_dow',
 'cos_dow',
 'is_business_hours',
 'is_midday_peak_window',
 'is_midday_weekday_peak',
 'slot_5min',
 'BU_TotActPwr_Tech_Room_lag_12',
 'BU_TotActPwr_Tech_Room_lag_24',
 'BU_TotActPwr_Tech_Room_lag_48',
 'BU_TotActPwr_Tech_Room_lag_96',
 'BU_TotActPwr_Tech_Room_lag_192',
 'BU_TotActPwr_Tech_Room_lag_288',
 'BU_TotActPwr_Tech_Room_roll_mean_24',
 'BU_TotActPwr_Tech_Room_roll_mean_48',
 'BU_TotActPwr_Tech_Room_roll_mean_96',
 'BU_TotActPwr_Tech_Room_roll_mean_288',
 'BU_TotActPwr_Tech_Room_roll_std_12',
 'BU_TotActPwr_Tech_Room_roll_std_24',
 'BU_TotActPwr_Tech_Room_roll_std_48',
 'BU_TotActPwr_Tech_Room_roll_std_96',
 'BU_TotActPwr_Tech_Room_trend_1_4',
 'BU_TotActPwr_Tech_Room_trend_1_12',
 'BU_TotActPwr_Tech_Room_trend_1_96',
 'BU_TotActPwr_Tech_Room_trend_96_192',
 'BU_TotActPwr_Tech_Room_yeste

## without resedulas

In [7]:
target_col = "BU_TotActPwr_Tech_Room"
feature_cols = [c for c in df_imputed.columns if c != target_col]

results_df, horizon_results = run_profile_training_experiment(
    df=df_imputed,
    target_col=target_col,
    feature_cols=feature_cols,
    dataset_name="15min_Imputed",
    feature_set_name="ALL_features",
    horizon_steps=672,
    issue_hour=23,
    issue_minute=45,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    selected_models=["ridge", "random_forest", "xgboost"],
    drop_feature_nan=False,
    data_mode="dense",
)

results_df

Training ridge for day-ahead profile forecasting: BU_TotActPwr_Tech_Room [dense]
Training random_forest for day-ahead profile forecasting: BU_TotActPwr_Tech_Room [dense]
Training xgboost for day-ahead profile forecasting: BU_TotActPwr_Tech_Room [dense]


,run_id,timestamp,task_type,dataset_name,feature_set_name,data_mode,target,model_name,model_params,n_features,...,test_MAE,test_RMSE,test_MAPE,test_sMAPE,test_R2,model_path,prediction_path,horizon_metrics_path,plot_path,params_path
0,run_20260428_110000_23120f5e,2026-04-28 11:00:00,day_ahead_profile_forecasting,15min_Imputed,ALL_features,dense,BU_TotActPwr_Tech_Room,ridge,"{""estimator__alpha"": 5.0, ""estimator__copy_X"":...",40,...,0.250219,0.403423,15.333984,16.038189,0.921017,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...
1,run_20260428_110004_1da2e590,2026-04-28 11:00:04,day_ahead_profile_forecasting,15min_Imputed,ALL_features,dense,BU_TotActPwr_Tech_Room,random_forest,"{""estimator__bootstrap"": true, ""estimator__ccp...",40,...,0.235844,0.386762,14.560480,14.380775,0.927406,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...
2,run_20260428_110859_55a1dc1a,2026-04-28 11:08:59,day_ahead_profile_forecasting,15min_Imputed,ALL_features,dense,BU_TotActPwr_Tech_Room,xgboost,"{""estimator__objective"": ""reg:squarederror"", ""...",40,...,0.246475,0.407661,15.142522,15.308701,0.919349,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...


# Data_Nans

    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",

In [105]:
path = r"C:\Data_analysis\Thesis\Data\03_Training\Data_with_NaNs\15_min\df_BU_TotActPwr_Tech_Room.parquet"
df_with_nans = pd.read_parquet(path)

In [106]:
df_with_nans.head()

,BU_TotActPwr_Tech_Room,BA_Soc,PV_WS_AirTemp,PV_WS_Radiation,dayofweek,is_weekend,hour,month,dayofmonth,minute_of_day,...,BU_TotActPwr_Tech_Room_roll_std_96,BU_TotActPwr_Tech_Room_trend_1_4,BU_TotActPwr_Tech_Room_trend_1_12,BU_TotActPwr_Tech_Room_trend_1_96,BU_TotActPwr_Tech_Room_trend_96_192,BU_TotActPwr_Tech_Room_yesterday_mean,BU_TotActPwr_Tech_Room_yesterday_max,BU_TotActPwr_Tech_Room_yesterday_midday_max,BU_TotActPwr_Tech_Room_yesterday_peak_time_step,BU_TotActPwr_Tech_Room_lastweek_same_day_peak
Time,,,,,,,,,,,,,,,,,,,,,
2025-10-15 00:00:00,3.62267,83.36667,128.33333,-4.53333,2,0,0,10,15,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10-15 00:15:00,3.62067,83.10000,127.00000,-4.36667,2,0,0,10,15,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10-15 00:30:00,3.68567,82.80000,126.33333,-4.43333,2,0,0,10,15,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10-15 00:45:00,3.68367,82.56667,125.00000,-4.70000,2,0,0,10,15,45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10-15 01:00:00,3.59833,82.26667,123.33333,-4.50000,2,0,1,10,15,60,...,NaN,0.061,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
target_col = "BU_TotActPwr_Tech_Room"
results_df_nan, horizon_results_nan = run_profile_training_experiment(
    df=df_with_nans,
    target_col=target_col,
    feature_cols=[c for c in df_with_nans.columns if c != target_col],
    dataset_name="15min_nans",
    feature_set_name="weather_features",
    horizon_steps=96,
    issue_hour=23,
    issue_minute=45,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    selected_models=["hist_gbr", "xgboost"],
    drop_feature_nan=False,
    data_mode="NaNs",
)

results_df_nan

Training hist_gbr for day-ahead profile forecasting: BU_TotActPwr_Tech_Room [NaNs]
Training xgboost for day-ahead profile forecasting: BU_TotActPwr_Tech_Room [NaNs]


,run_id,timestamp,task_type,dataset_name,feature_set_name,data_mode,target,model_name,model_params,n_features,...,test_MAE,test_RMSE,test_MAPE,test_sMAPE,test_R2,model_path,prediction_path,horizon_metrics_path,plot_path,params_path
0,run_20260410_172038_4b7b1df9,2026-04-10 17:20:38,day_ahead_profile_forecasting,15min_nans,weather_features,NaNs,BU_TotActPwr_Tech_Room,hist_gbr,"{""estimator__categorical_features"": ""from_dtyp...",48,...,0.318112,0.459920,21.613918,21.217906,0.897199,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...
1,run_20260410_172143_3c35a617,2026-04-10 17:21:43,day_ahead_profile_forecasting,15min_nans,weather_features,NaNs,BU_TotActPwr_Tech_Room,xgboost,"{""estimator__objective"": ""reg:squarederror"", ""...",48,...,0.203547,0.363876,12.610058,12.871555,0.935652,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...,C:\Data_analysis\Thesis\outputs\profile_foreca...


# Plots

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_profile_day(prediction_csv_path, row_idx=0, title=None):
    df = pd.read_csv(prediction_csv_path, index_col=0, parse_dates=True)

    true_cols = [c for c in df.columns if c.endswith("_true")]
    pred_cols = [c for c in df.columns if c.endswith("_pred")]

    true_vals = df.iloc[row_idx][true_cols].values.astype(float)
    pred_vals = df.iloc[row_idx][pred_cols].values.astype(float)

    issue_time = df.index[row_idx]

    plt.figure(figsize=(14, 5))
    plt.plot(true_vals, label="Actual")
    plt.plot(pred_vals, label="Predicted")
    plt.xlabel("15-minute step of next day")
    plt.ylabel("Load / Power")
    plt.title(title or f"Day-ahead profile forecast\nIssue time: {issue_time}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
prediction_csv_path = r"C:\Data_analysis\Thesis\JupyterNotebooks\artifacts\profile_forecasting\predictions\multi_xgboost_BU_TotActPwr_Tech_Room_profile_predictions.csv"
plot_profile_day(prediction_csv_path, row_idx=0)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_multiple_profile_days(prediction_csv_path, row_indices=(0, 1, 2)):
    df = pd.read_csv(prediction_csv_path, index_col=0, parse_dates=True)

    true_cols = [c for c in df.columns if c.endswith("_true")]
    pred_cols = [c for c in df.columns if c.endswith("_pred")]

    for row_idx in row_indices:
        true_vals = df.iloc[row_idx][true_cols].values.astype(float)
        pred_vals = df.iloc[row_idx][pred_cols].values.astype(float)
        issue_time = df.index[row_idx]

        plt.figure(figsize=(14, 5))
        plt.plot(true_vals, label="Actual")
        plt.plot(pred_vals, label="Predicted")
        plt.xlabel("15-minute step of next day")
        plt.ylabel("Load / Power")
        plt.title(f"Day-ahead profile forecast\nIssue time: {issue_time}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_multiple_profile_days(
    prediction_csv_path,
    row_indices=(0, 5, 10)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_average_profile(prediction_csv_path, title="Average day-ahead profile"):
    df = pd.read_csv(prediction_csv_path, index_col=0, parse_dates=True)

    true_cols = [c for c in df.columns if c.endswith("_true")]
    pred_cols = [c for c in df.columns if c.endswith("_pred")]

    true_mean = df[true_cols].mean(axis=0).values.astype(float)
    pred_mean = df[pred_cols].mean(axis=0).values.astype(float)

    plt.figure(figsize=(14, 5))
    plt.plot(true_mean, label="Average Actual")
    plt.plot(pred_mean, label="Average Predicted")
    plt.xlabel("15-minute step of next day")
    plt.ylabel("Load / Power")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_average_profile(prediction_csv_path)

In [ ]:
import pandas as pd
import numpy as np


def load_profile_predictions_as_timeseries(prediction_csv_path: str) -> pd.DataFrame:

    df = pd.read_csv(prediction_csv_path, index_col=0, parse_dates=True)

    true_cols = sorted([c for c in df.columns if c.endswith("_true")])
    pred_cols = sorted([c for c in df.columns if c.endswith("_pred")])

    if len(true_cols) != len(pred_cols):
        raise ValueError("Mismatch between number of true and predicted columns.")

    records = []

    for issue_time, row in df.iterrows():
        for i, (t_col, p_col) in enumerate(zip(true_cols, pred_cols), start=1):
            forecast_time = issue_time + pd.Timedelta(minutes=15 * i)

            actual = float(row[t_col])
            predicted = float(row[p_col])

            records.append({
                "forecast_time": forecast_time,
                "issue_time": issue_time,
                "step": i,
                "actual": actual,
                "predicted": predicted,
                "residual": actual - predicted,
                "abs_error": abs(actual - predicted),
            })

    ts_df = pd.DataFrame(records).sort_values("forecast_time")
    ts_df = ts_df.set_index("forecast_time")

    return ts_df

In [ ]:
import matplotlib.pyplot as plt


def plot_profile_diagnostics(
    prediction_csv_path: str,
    model_name: str = "profile_model",
    target_name: str = "target",
    dataset_name: str = "dataset",
    zoom_start: str | None = None,
    zoom_end: str | None = None,
    figsize=(16, 16),
):
    """
    Create a 4-panel diagnostic plot similar to your screenshot:
    1. Actual vs Predicted over time
    2. Residuals over time
    3. Zoomed window
    4. Actual vs Predicted scatter
    """
    ts_df = load_profile_predictions_as_timeseries(prediction_csv_path)

    if zoom_start is None:
        zoom_start = str(ts_df.index.min())
    if zoom_end is None:
        zoom_end = str(min(ts_df.index.min() + pd.Timedelta(days=3), ts_df.index.max()))

    zoom_df = ts_df.loc[zoom_start:zoom_end].copy()

    fig, axes = plt.subplots(4, 1, figsize=figsize)

    title_prefix = f"{model_name} - {target_name} [{dataset_name}]"

    # 1. Actual vs predicted over time
    axes[0].plot(ts_df.index, ts_df["actual"], label="Actual")
    axes[0].plot(ts_df.index, ts_df["predicted"], label="Predicted")
    axes[0].set_title(title_prefix)
    axes[0].set_ylabel(f"{target_name} [kW]")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 2. Residuals over time
    axes[1].plot(ts_df.index, ts_df["residual"])
    axes[1].axhline(0, linestyle="--")
    axes[1].set_title(f"{title_prefix} - Residuals Over Time")
    axes[1].set_ylabel("Residual [kW]")
    axes[1].grid(True, alpha=0.3)

    # 3. Zoomed window
    axes[2].plot(zoom_df.index, zoom_df["actual"], label="Actual")
    axes[2].plot(zoom_df.index, zoom_df["predicted"], label="Predicted")
    axes[2].set_title(f"{title_prefix} - Zoomed Window")
    axes[2].set_ylabel(f"{target_name} [kW]")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    # 4. Scatter
    axes[3].scatter(ts_df["actual"], ts_df["predicted"], alpha=0.5)
    min_val = min(ts_df["actual"].min(), ts_df["predicted"].min())
    max_val = max(ts_df["actual"].max(), ts_df["predicted"].max())
    axes[3].plot([min_val, max_val], [min_val, max_val], linestyle="--")
    axes[3].set_title(f"{title_prefix} - Actual vs Predicted Scatter")
    axes[3].set_xlabel("Actual [kW]")
    axes[3].set_ylabel("Predicted [kW]")
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
prediction_csv_path = r"C:\Data_analysis\Thesis\JupyterNotebooks\artifacts\profile_forecasting\predictions\multi_xgboost_BU_TotActPwr_Tech_Room_profile_predictions.csv"

plot_profile_diagnostics(
    prediction_csv_path=prediction_csv_path,
    model_name="multi_xgboost",
    target_name="BU_TotActPwr_Tech_Room",
    dataset_name="15min_profile_data",
    zoom_start="2026-01-29",
    zoom_end="2026-02-02",
)

In [ ]:

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch version:", torch.__version__)
print("Device:", device)
